## 01 - Scraping

Downloads the public reviews of the Trenord app from Google Play and saves them as they are.

In [2]:
# --- IMPORTS ---

# google_play_scraper gives us two functions:
# - app()     → fetches general app info (name, global rating, installs, etc.)
# - reviews() → fetches user reviews page by page
# - Sort      → an enum with sorting options: NEWEST, MOST_RELEVANT, RATING
from google_play_scraper import app, reviews, Sort
import pandas as pd
import time

In [3]:
# --- CONFIGURATION ---
# All settings are defined here at the top so that if you want
# to reuse this notebook for a different app, you only need to
# change values in this cell - nothing else in the notebook changes.

# The unique package name of the app on the Play Store.
# You can find it in the Play Store URL:
# https://play.google.com/store/apps/details?id=it.nordcom.app&hl=en
APP_ID = "it.nordcom.app"

# The language used to query the Play Store interface.
# "it" = Italian. Note: this does NOT filter out reviews in other
# languages - reviews in English etc. will still appear.
# We will handle language detection later in the cleaning step.
LANGUAGE = "it"

# The country from which we simulate the request.
COUNTRY = "it"

# How many reviews to fetch per request (per "page").
# 200 is the maximum reliable batch size supported by the library.
BATCH_SIZE = 200

# Total number of reviews we want to download.
# Set to None if you want to download ALL available reviews.
# Warning: setting None can take a very long time for popular apps.
MAX_REVIEWS = 16000

In [4]:
# --- STEP 1: FETCH APP METADATA ---
# Before downloading reviews, we fetch general information about the app.
# This gives us context: the global rating, number of installs, current version, etc.
# The app() function returns a dictionary with all the app's public information.

app_info = app(
    APP_ID,         # the package name we defined in the configuration cell
    lang=LANGUAGE,  # language for the Play Store interface
    country=COUNTRY # country from which we simulate the request
)

# Print the most useful fields so we can see what we are working with.
print(f"App name   : {app_info['title']}")
print(f"Developer  : {app_info['developer']}")
print(f"Rating     : {app_info['score']:.2f} / 5.0")  # :.2f means 2 decimal places
print(f"Ratings    : {app_info['ratings']:,}")        # :, adds thousands separator
print(f"Installs   : {app_info['installs']}")
print(f"Version    : {app_info['version']}")
print(f"Updated    : {app_info['updated']}")

App name   : Trenord - Orari e Info Treni
Developer  : Trenord
Rating     : 1.82 / 5.0
Ratings    : 16,787
Installs   : 1.000.000+
Version    : 4.16.7
Updated    : 1785400777


In [5]:
# --- STEP 2: SCRAPE REVIEWS ---
# The Play Store does not return all reviews at once.
# It works like a book with pages: you read one page, then ask for the next one.
# Each "page" contains up to BATCH_SIZE reviews.
# The API returns a "continuation token" with each page - this is just a string
# that tells the API where we left off, so it knows which page to give us next.
# When the token is None, there are no more pages and we stop.

# This list will accumulate all reviews across all pages
all_reviews = []
continuation_token = None
batch_number = 0

print(f"Scraping {APP_ID} | target: {MAX_REVIEWS} reviews | batch size: {BATCH_SIZE}")
print("-" * 50)

while True:
    batch_number += 1
    remaining = MAX_REVIEWS - len(all_reviews)

    if remaining <= 0:
        print(f"Reached target of {MAX_REVIEWS} reviews.")
        break

    batch_result, continuation_token = reviews(
        APP_ID,
        lang=LANGUAGE,
        country=COUNTRY,
        sort=Sort.NEWEST,
        count=min(BATCH_SIZE, remaining),
        continuation_token=continuation_token
    )

    if not batch_result:
        print(f"No more reviews available after batch {batch_number}.")
        break

    all_reviews.extend(batch_result)
    print(f"Batch {batch_number:3d} | fetched: {len(batch_result):3d} | total: {len(all_reviews):,}")

    if continuation_token is None:
        print("Last page reached.")
        break

    time.sleep(0.5)

print("-" * 50)
print(f"Done. Total reviews collected: {len(all_reviews):,}")

Scraping it.nordcom.app | target: 16000 reviews | batch size: 200
--------------------------------------------------
Batch   1 | fetched: 200 | total: 200
Batch   2 | fetched: 200 | total: 400
Batch   3 | fetched: 200 | total: 600
Batch   4 | fetched: 200 | total: 800
Batch   5 | fetched: 200 | total: 1,000
Batch   6 | fetched: 200 | total: 1,200
Batch   7 | fetched: 200 | total: 1,400
Batch   8 | fetched: 200 | total: 1,600
Batch   9 | fetched: 200 | total: 1,800
Batch  10 | fetched: 200 | total: 2,000
Batch  11 | fetched: 200 | total: 2,200
Batch  12 | fetched: 200 | total: 2,400
Batch  13 | fetched: 200 | total: 2,600
Batch  14 | fetched: 200 | total: 2,800
Batch  15 | fetched: 200 | total: 3,000
Batch  16 | fetched: 200 | total: 3,200
Batch  17 | fetched: 200 | total: 3,400
Batch  18 | fetched: 200 | total: 3,600
Batch  19 | fetched: 200 | total: 3,800
Batch  20 | fetched: 200 | total: 4,000
Batch  21 | fetched: 200 | total: 4,200
Batch  22 | fetched: 200 | total: 4,400
Batch  23 |

In [6]:
# --- COVERAGE ---
# How much of the app's total reviews did we actually get?
total_ratings = app_info["ratings"]
coverage = len(all_reviews) / total_ratings * 100

print(f"Collected  : {len(all_reviews):,} reviews")
print(f"Store total: {total_ratings:,} ratings")
print(f"Coverage   : {coverage:.1f}%")

Collected  : 7,329 reviews
Store total: 16,787 ratings
Coverage   : 43.7%


### Data completeness

The scraper collected every written review it could reach: it stopped because Google Play returned no more reviews, not because it hit the target. The store's count of 16,787 ratings also includes star-only ratings, which have no text and cannot be downloaded. So the coverage printed above is not the share of written reviews that were collected. Google Play is the only source, so the results describe only the users who review the app there.

In [ ]:
# --- STEP 3: BUILD THE DATAFRAME ---
# all_reviews is a list of dictionaries - one dictionary per review.
# Each dictionary has many fields, but we only keep the ones that are useful.
# Here is what each field contains:
# - reviewId            : unique identifier for the review
# - userName            : the author's display name
# - score               : the rating given by the user (integer from 1 to 5)
# - content             : the actual review text written by the user
# - thumbsUpCount       : how many users found this review helpful
# - appVersion          : the app version installed when the review was written
# - at                  : the date and time the review was posted (datetime object)
# - replyContent        : the developer's reply text (None if no reply)
# - repliedAt           : the date and time of the developer's reply (None if no reply)

# userImage is not kept; reviewCreatedVersion duplicates appVersion.

df = pd.DataFrame([
    {
        "review_id"  : r["reviewId"],
        "author"     : r["userName"],
        "rating"     : r["score"],
        "text"       : r["content"],
        "thumbs_up"  : r["thumbsUpCount"],
        "app_version": r.get("appVersion"),
        "date"       : r["at"],
        "has_reply"  : r["replyContent"] is not None,
        "reply_text" : r["replyContent"],
        "reply_date" : r.get("repliedAt"),
    }
    for r in all_reviews
])

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")

In [10]:
# --- STEP 4: SAVE RAW DATA ---
# We save the raw dataframe immediately after scraping.
# This way if the kernel restarts we don't need to scrape again -
# we can just reload the CSV and continue from here.

# "raw" means the data is unprocessed - exactly as it came from the Play Store.
# We will create a separate "clean" version later.
df.to_csv("../data/raw/trenord_reviews_raw.csv", index=False, encoding="utf-8-sig")
# utf-8-sig adds a BOM (Byte Order Mark) so Excel opens it correctly
# index=False means we don't save the row numbers as a column

print(f"Raw data saved! {len(df)} reviews saved to ../data/raw/trenord_reviews_raw.csv")

Raw data saved! 7329 reviews saved to ../data/raw/trenord_reviews_raw.csv
